In [1]:
import os
from dotenv import load_dotenv

# Modern LangChain Imports
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_astradb import AstraDBVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

load_dotenv("/home/abhi/AI_Workspace/personal/Generative-AI-Engineer-Portfolio/.env")  # Load environment variables from .env file

/home/abhi/.virtualenvs/my_genai_venv/lib/python3.11/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


True

In [2]:
# Azure OpenAI Keys
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_LLM_MODEL = os.getenv("AZURE_OPENAI_LLM_MODEL")
AZURE_OPENAI_EMBEDDINGS_MODEL = os.getenv("AZURE_OPENAI_EMBEDDINGS_MODEL")

# Astra DB
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_COLLECTION = "langchain_pdf_rag"

In [3]:
#Load PDF and Split into Chunks
loader = PyPDFLoader("budget_speech.pdf")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(raw_docs)

In [4]:
# Initialize Azure OpenAI Embeddings
embedding_model = AzureOpenAIEmbeddings(
    api_key = AZURE_OPENAI_API_KEY,
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    model = AZURE_OPENAI_EMBEDDINGS_MODEL,
    api_version = "2024-12-01-preview",
    dimensions = 1024
)
embedding_model

AzureOpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7a75b5f46750>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7a75b5e187d0>, model='text-embedding-3-large', dimensions=1024, deployment=None, openai_api_version='2024-12-01-preview', openai_api_base=None, openai_api_type='azure', openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=2048, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True, azure_endpoint='https://pfs-2-namit-resource.cognitiveservices.azure.com/', azure_ad_token=None, azure_ad_token_provider=None, azure_ad_async_token_provider=None, valida

In [5]:
#Intialize Vector Store
vector_store = AstraDBVectorStore(
    collection_name=ASTRA_DB_COLLECTION,
    embedding=embedding_model,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN
)

vector_store

In [6]:
#Create embeddings and store to Vector Store
vector_store.add_documents(documents=split_docs)

#Create retriever from vector store
retriever = vector_store.as_retriever(kwargs={"k":4})
retriever

VectorStoreRetriever(tags=['AstraDBVectorStore', 'AzureOpenAIEmbeddings'], vectorstore=<langchain_astradb.vectorstores.AstraDBVectorStore object at 0x7a75b5e19b50>, search_kwargs={})

In [7]:
llm = AzureChatOpenAI(
    model=AZURE_OPENAI_LLM_MODEL,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2025-01-01-preview"
)
llm

AzureChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7a75b5db3bd0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7a75b645b910>, root_client=<openai.lib.azure.AzureOpenAI object at 0x7a75bc416ed0>, root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x7a75b5e19950>, model_name='gpt-5.2-chat', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, disabled_params={'parallel_tool_calls': None}, azure_endpoint='https://pfs-2-namit-resource.cognitiveservices.azure.com/', openai_api_version='2025-01-01-preview', openai_api_type='azure')

In [8]:
#Define prompt template for llm
prompt = ChatPromptTemplate.from_template(
    """
    You are an AI Assistant that answer's user queries based on the provided context.
    
    Question:
    {question}

    Context:
    {context}
    """  
)

prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n    You are an AI Assistant that answer's user queries based on the provided context.\n\n    Question:\n    {question}\n\n    Context:\n    {context}\n    "), additional_kwargs={})])

In [9]:
#Function to join content of all relevant context fetched
def concat_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [10]:
#Create final chain
rag_chain = (
    {
        "context": retriever | concat_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain

{
  context: VectorStoreRetriever(tags=['AstraDBVectorStore', 'AzureOpenAIEmbeddings'], vectorstore=<langchain_astradb.vectorstores.AstraDBVectorStore object at 0x7a75b5e19b50>, search_kwargs={})
           | RunnableLambda(concat_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n    You are an AI Assistant that answer's user queries based on the provided context.\n\n    Question:\n    {question}\n\n    Context:\n    {context}\n    "), additional_kwargs={})])
| AzureChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7a75b5db3bd0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7a75b645b910>, root_client=<openai.lib.azure.AzureOpenAI object at 0x7a75bc416ed0>, root_asyn

In [11]:
first_ques = True
while True:
    if first_ques:
        print("Hi! I'm an helpful AI Assistant.\nI'll help you answer questions based on you file - budget_speech.pdf")
        query_text = input("Type your Question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("What's your next question (or type 'quit' to exit): ").strip()
    
    if query_text.lower() == "quit":
        break
    elif query_text == "" or query_text == " ":
        print("Please Enter a Question\n")
        continue

    first_ques = False

    print("---------------------------------------")
    print(f"YOU: {query_text}")

    answer = rag_chain.invoke(query_text)
    
    print(f"ANSWER: {answer}")
    print("---------------------------------------")   


Hi! I'm an helpful AI Assistant.
I'll help you answer questions based on you file - budget_speech.pdf
---------------------------------------
YOU: Is there any update on GST?
ANSWER: Based on the **provided context**, the update on **GST** is as follows:

- **Legislative changes in GST laws are proposed**, subject to notification after coordination with States and based on **recommendations of the GST Council**.
- A key change mentioned is an **amendment to Section 15 of the CGST Act, 2017**, specifically relating to **post‑sale discounts**.
- These amendments are **not yet in force** and will become effective **from a date to be notified**.
- The context does **not indicate any change in GST rates**, but focuses on **valuation-related provisions** (post‑sale discounts).

So, in summary, **yes—there is an update**, but it is at the level of **proposed legislative amendments**, awaiting notification and implementation rather than immediate operational changes.
--------------------------